# Fase II Pipeline ETL Distribuido
**Proyecto Integrador:** Sistema Analítico y Predictivo de Anomalías Térmicas Continentales - Grupo 6

**Integrantes:** Christian Salinas, Mateo Castillo

**Dataset:** Red INMET con datos climáticos de Brasil y Latinoamérica

## 1. Diseño de Arquitectura del Pipeline
Para este proyecto manejamos una arquitectura por capas sencilla y eficiente:
* **Fuente de datos:** Archivos CSV de la Red INMET guardados en Google Drive.
* **Ingesta Raw:** Carga distribuida utilizando PySpark inferiendo esquemas.
* **Limpieza Bronze:** Eliminación de duplicados y tratamiento de nulos en variables clave.
* **Transformación Silver:** Estandarización de texto, Feature Engineering para la depresión del punto de rocío y normalización Z-score.
* **Almacenamiento:** Exportación a formato Parquet con compresión Snappy y particionamiento por año, listo para la Fase III.

## 2. Ingesta de Datos y Validación Inicial
Cargamos los datos desde la fuente original en CSV usando PySpark y revisamos la calidad inicial del dataset crudo midiendo nulos, duplicados y esquemas.

In [1]:
!pip install pyspark --quiet
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum, isnan, when, round as spark_round, upper
import time

spark = SparkSession.builder.appName("Fase_II_ETL").master("local[*]").getOrCreate()

from google.colab import drive
drive.mount('/content/drive')

ruta_csv = '/content/drive/MyDrive/Proyecto_Clima/Dataset_LatAm/*.csv'
df_raw = spark.read.csv(ruta_csv, header=True, inferSchema=True, sep=",")
df_raw = df_raw.replace(-9999, None)

total_filas = df_raw.count()
print("=== ESTADO CRUDO RAW ===")
print(f"Filas totales: {total_filas:,}")
print(f"Columnas: {len(df_raw.columns)}\n")

print("Validación de Nulos Inicial:")
exprs_nulos = []
for c in df_raw.columns:
    if df_raw.schema[c].dataType.typeName() in ['double', 'float']:
        condicion = col(f"`{c}`").isNull() | isnan(col(f"`{c}`"))
    else:
        condicion = col(f"`{c}`").isNull()
    exprs_nulos.append(spark_round((spark_sum(when(condicion, 1).otherwise(0)) / total_filas) * 100, 2).alias(c))

df_raw.select(exprs_nulos).show(truncate=False)

duplicados_iniciales = total_filas - df_raw.dropDuplicates().count()
print(f"Duplicados exactos encontrados: {duplicados_iniciales:,}\n")

Mounted at /content/drive
=== ESTADO CRUDO RAW ===
Filas totales: 27,354,355
Columnas: 14

Validación de Nulos Inicial:
+------+---+---+---+----+------------+-------+--------+----------------------+----------------+-------+-------------+------------+----------------+
|Estado|ano|mes|dia|hora|PRECIPITA��O|PRESSAO|RADIACAO|TEMPERATURA.BULBO.SECO|PONTO.DE.ORVALHO|UMIDADE|VENTO.DIRE��O|VENTO.RAJADA|VENTO.VELOCIDADE|
+------+---+---+---+----+------------+-------+--------+----------------------+----------------+-------+-------------+------------+----------------+
|0.0   |0.0|0.0|0.0|0.0 |0.0         |0.0    |0.0     |0.0                   |0.0             |0.0    |0.0          |0.0         |0.0             |
+------+---+---+---+----+------------+-------+--------+----------------------+----------------+-------+-------------+------------+----------------+

Duplicados exactos encontrados: 0



## 3. Limpieza de Datos
Definimos una función de auditoría para rastrear la pérdida de registros. Procedemos a eliminar duplicados exactos y descartar registros sin datos en la variable objetivo de temperatura.

In [2]:
def log_etl(df, paso_nombre, filas_previas):
    filas_actuales = df.count()
    cols_actuales = len(df.columns)
    perdida = filas_previas - filas_actuales
    print(f"► {paso_nombre}")
    print(f"  Filas resultantes: {filas_actuales:,} | Registros filtrados: {perdida:,} | Columnas: {cols_actuales}\n")
    return filas_actuales

filas_tracker = total_filas

df_t1 = df_raw.dropDuplicates()
df_t1 = df_t1.dropna(subset=["`TEMPERATURA.BULBO.SECO`"])
filas_tracker = log_etl(df_t1, "Limpieza de nulos en variable principal y duplicados", filas_tracker)

► Limpieza de nulos en variable principal y duplicados
  Filas resultantes: 27,354,355 | Registros filtrados: 0 | Columnas: 14



## 4. Transformación de Datos
Aplicamos estandarización de columnas para evitar errores de sintaxis en Spark, normalizamos el texto de las regiones y aplicamos Feature Engineering creando un indicador de estrés térmico.

In [8]:
from pyspark.ml.feature import StandardScaler, VectorAssembler

# Estandarización de columnas y texto
nuevas_columnas = [c.replace(".", "_") for c in df_t1.columns]
df_t2 = df_t1.toDF(*nuevas_columnas)
df_t2 = df_t2.withColumn("Estado", upper(col("Estado")))

# Feature Engineering variable nueva completamente limpia
df_t3 = df_t2.withColumn("DEPRESION_PUNTO_ORVALHO", col("TEMPERATURA_BULBO_SECO") - col("PONTO_DE_ORVALHO"))

# Normalización Z-score
assembler = VectorAssembler(inputCols=["TEMPERATURA_BULBO_SECO"], outputCol="temp_vector", handleInvalid="skip")
df_ensamblado = assembler.transform(df_t3)
scaler = StandardScaler(inputCol="temp_vector", outputCol="TEMPERATURA_NORMALIZADA", withStd=True, withMean=True)
scaler_model = scaler.fit(df_ensamblado)
df_t4 = scaler_model.transform(df_ensamblado).drop("temp_vector")

filas_tracker = log_etl(df_t4, "Transformaciones completas aplicadas", filas_tracker)

► Transformaciones completas aplicadas
  Filas resultantes: 27,354,355 | Registros filtrados: 0 | Columnas: 16



In [9]:
# Verificacion visual de las variables para la evidencia del informe
print("=== MUESTRA DE VARIABLES TRANSFORMADAS ===")
df_t4.select(
    "TEMPERATURA_BULBO_SECO",
    "TEMPERATURA_NORMALIZADA",
    "PONTO_DE_ORVALHO",
    "DEPRESION_PUNTO_ORVALHO"
).show(10)

# Resumen estadistico de la nueva variable de ingenieria de caracteristicas
print("=== ESTADISTICAS DE LA DEPRESION DEL PUNTO DE ROCIO ===")
df_t4.select("DEPRESION_PUNTO_ORVALHO").summary("count", "mean", "min", "max").show()

=== MUESTRA DE VARIABLES TRANSFORMADAS ===
+----------------------+-----------------------+----------------+-----------------------+
|TEMPERATURA_BULBO_SECO|TEMPERATURA_NORMALIZADA|PONTO_DE_ORVALHO|DEPRESION_PUNTO_ORVALHO|
+----------------------+-----------------------+----------------+-----------------------+
|                  23.8|   [-0.2826063440409...|            12.6|     11.200000000000001|
|                  22.5|   [-0.5214702289158...|            12.2|                   10.3|
|                  18.3|   [-1.2931843185115...|            13.7|      4.600000000000001|
|                  21.7|   [-0.6684633888388...|            11.8|      9.899999999999999|
|                  21.6|   [-0.686837533829206]|             9.3|                   12.3|
|                  18.4|   [-1.2748101735211...|            10.3|      8.099999999999998|
|                  24.6|   [-0.1356131841180...|            10.5|     14.100000000000001|
|                  26.2|   [0.15837313572797...|         

## 5. Almacenamiento y Validación Final
Guardamos el dataset procesado y limpio directamente en Google Drive utilizando formato Parquet con compresión Snappy y particionado por año. Finalmente verificamos la integridad de los datos cargados.

In [10]:
ruta_salida = "/content/drive/MyDrive/Proyecto_Clima/dataset_clima_etl_parquet"

inicio_escritura = time.time()
df_t4.write.mode("overwrite") \
    .option("compression", "snappy") \
    .partitionBy("ano") \
    .parquet(ruta_salida)
tiempo_load = time.time() - inicio_escritura

df_final = spark.read.parquet(ruta_salida)

print("=== REPORTE DE VALIDACIÓN FINAL ===")
print(f"Filas origen: {total_filas:,} -> Filas destino: {df_final.count():,}")
print(f"Columnas origen: {len(df_raw.columns)} -> Columnas destino: {len(df_final.columns)}")
print(f"Tiempo de escritura distribuida: {tiempo_load:.2f} segundos")
print("Validación de tipos post-procesamiento:")
df_final.printSchema()
print("Pipeline ETL ejecutado y validado exitosamente.")

=== REPORTE DE VALIDACIÓN FINAL ===
Filas origen: 27,354,355 -> Filas destino: 27,354,355
Columnas origen: 14 -> Columnas destino: 16
Tiempo de escritura distribuida: 512.70 segundos
Validación de tipos post-procesamiento:
root
 |-- Estado: string (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia: integer (nullable = true)
 |-- hora: integer (nullable = true)
 |-- PRECIPITA��O: double (nullable = true)
 |-- PRESSAO: double (nullable = true)
 |-- RADIACAO: double (nullable = true)
 |-- TEMPERATURA_BULBO_SECO: double (nullable = true)
 |-- PONTO_DE_ORVALHO: double (nullable = true)
 |-- UMIDADE: integer (nullable = true)
 |-- VENTO_DIRE��O: double (nullable = true)
 |-- VENTO_RAJADA: double (nullable = true)
 |-- VENTO_VELOCIDADE: double (nullable = true)
 |-- DEPRESION_PUNTO_ORVALHO: double (nullable = true)
 |-- TEMPERATURA_NORMALIZADA: vector (nullable = true)
 |-- ano: integer (nullable = true)

Pipeline ETL ejecutado y validado exitosamente.
